In [1]:
print("Colab ready")

Colab ready


In [2]:
!pip install -q rdkit

import rdkit
import torch
from rdkit import Chem

print("RDKit version:", rdkit.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

RDKit version: 2026.09.2
PyTorch version: 2.14.1+cpu
CUDA available: False


In [3]:
import pandas as pd

url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"
df = pd.read_csv(url)

print("Total rows:", len(df))
print("Columns:", list(df.columns))
print("\nSR-p53 label breakdown (including missing values):")
print(df["SR-p53"].value_counts(dropna=False))

Total rows: 7831
Columns: ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53', 'mol_id', 'smiles']

SR-p53 label breakdown (including missing values):
SR-p53
0.0    6351
NaN    1057
1.0     423
Name: count, dtype: int64


In [4]:
# Keep only compounds with measured SR-p53 labels
p53_df = df[["mol_id", "smiles", "SR-p53"]].dropna(subset=["SR-p53"]).copy()
p53_df["SR-p53"] = p53_df["SR-p53"].astype(int)

# Convert SMILES strings into RDKit Mol objects and remove invalid entries
p53_df["mol"] = p53_df["smiles"].apply(Chem.MolFromSmiles)
invalid_count = p53_df["mol"].isna().sum()
p53_df = p53_df.dropna(subset=["mol"]).reset_index(drop=True)

print(f"Dropped {invalid_count} invalid SMILES.")
print(f"Clean SR-p53 dataset size: {len(p53_df)}")
print(f"Active (1) count: {(p53_df['SR-p53'] == 1).sum()}")
print(f"Inactive (0) count: {(p53_df['SR-p53'] == 0).sum()}")
print(f"Positive rate: {p53_df['SR-p53'].mean():.2%}")
p53_df[["mol_id", "smiles", "SR-p53"]].head()


Dropped 7 invalid SMILES.
Clean SR-p53 dataset size: 6767
Active (1) count: 423
Inactive (0) count: 6344
Positive rate: 6.25%


,mol_id,smiles,SR-p53
0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1,0
1,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O,0
2,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C,0
3,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O,0
4,TOX5110,CC(C)(C)OOC(C)(C)CCC(C)(C)OOC(C)(C)C,0


In [5]:
import numpy as np
from rdkit.Chem import rdFingerprintGenerator

# Generate 2048-bit Morgan fingerprints (radius=2, equivalent to ECFP4)
mfp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

X = np.array(
    [mfp_gen.GetFingerprintAsNumPy(mol) for mol in p53_df["mol"]],
    dtype=np.float32,
)
y = p53_df["SR-p53"].to_numpy(dtype=np.int64)

print("Feature matrix X shape:", X.shape)
print("Target vector y shape:", y.shape)
print("X dtype:", X.dtype, "| min/max values:", X.min(), X.max())
print("Average active bits per molecule:", X.sum(axis=1).mean().round(2))


Feature matrix X shape: (6767, 2048)
Target vector y shape: (6767,)
X dtype: float32 | min/max values: 0.0 1.0
Average active bits per molecule: 28.88


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42
    ),
}

for name, clf in models.items():
  roc_scores = []
  pr_scores = []

  for train_idx, val_idx in skf.split(X, y):
    clf.fit(X[train_idx], y[train_idx])
    probs = clf.predict_proba(X[val_idx])[:, 1]
    roc_scores.append(roc_auc_score(y[val_idx], probs))
    pr_scores.append(average_precision_score(y[val_idx], probs))

  print(
      f"{name:20s} | 5-fold ROC-AUC: {np.mean(roc_scores):.4f} +/-"
      f" {np.std(roc_scores):.4f} | PR-AUC: {np.mean(pr_scores):.4f} +/-"
      f" {np.std(pr_scores):.4f}"
  )

LogisticRegression   | 5-fold ROC-AUC: 0.7784 +/- 0.0307 | PR-AUC: 0.3153 +/- 0.0400
RandomForest         | 5-fold ROC-AUC: 0.8590 +/- 0.0205 | PR-AUC: 0.4194 +/- 0.0311


In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42
    ),
}

for name, clf in models.items():
  roc_scores = []
  pr_scores = []

  for train_idx, val_idx in skf.split(X, y):
    clf.fit(X[train_idx], y[train_idx])
    probs = clf.predict_proba(X[val_idx])[:, 1]
    roc_scores.append(roc_auc_score(y[val_idx], probs))
    pr_scores.append(average_precision_score(y[val_idx], probs))

  print(
      f"{name:20s} | 5-fold ROC-AUC: {np.mean(roc_scores):.4f} +/-"
      f" {np.std(roc_scores):.4f} | PR-AUC: {np.mean(pr_scores):.4f} +/-"
      f" {np.std(pr_scores):.4f}"
  )

NameError: name 'X' is not defined

In [2]:
!pip install -q rdkit

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import torch

print("RDKit:", Chem.rdBase.rdkitVersion, "| PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

RDKit: 2026.09.2 | PyTorch: 2.14.1+cu130
CUDA available: True


In [3]:
url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"
df = pd.read_csv(url)

p53_df = df[["mol_id", "smiles", "SR-p53"]].dropna(subset=["SR-p53"]).copy()
p53_df["SR-p53"] = p53_df["SR-p53"].astype(int)
p53_df["mol"] = p53_df["smiles"].apply(Chem.MolFromSmiles)
p53_df = p53_df.dropna(subset=["mol"]).reset_index(drop=True)

mfp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
X = np.array(
    [mfp_gen.GetFingerprintAsNumPy(mol) for mol in p53_df["mol"]],
    dtype=np.float32,
)
y = p53_df["SR-p53"].to_numpy(dtype=np.int64)

print("Ready -> X:", X.shape, "| y:", y.shape, "| Positives:", y.sum())

Ready -> X: (6767, 2048) | y: (6767,) | Positives: 423


In [4]:
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)


class FingerprintMLP(nn.Module):

  def __init__(self, in_dim=2048):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(in_dim, 512),
        nn.BatchNorm1d(512),
        nn.SiLU(),
        nn.Dropout(0.3),
        nn.Linear(512, 128),
        nn.BatchNorm1d(128),
        nn.SiLU(),
        nn.Dropout(0.2),
        nn.Linear(128, 1),
    )

  def forward(self, x):
    return self.net(x).squeeze(-1)


torch.manual_seed(42)
np.random.seed(42)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
roc_scores, pr_scores = [], []

for train_idx, val_idx in skf.split(X, y):
  X_tr, y_tr = X[train_idx], y[train_idx].astype(np.float32)
  X_va, y_va = X[val_idx], y[val_idx]

  n_pos = max(1.0, float(y_tr.sum()))
  n_neg = max(1.0, float(len(y_tr) - y_tr.sum()))
  pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=device)

  train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
  train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

  model = FingerprintMLP(in_dim=X.shape[1]).to(device)
  optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
  criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

  for epoch in range(25):
    model.train()
    for bx, by in train_loader:
      bx, by = bx.to(device), by.to(device)
      optimizer.zero_grad()
      loss = criterion(model(bx), by)
      loss.backward()
      optimizer.step()

  model.eval()
  with torch.no_grad():
    val_logits = model(torch.from_numpy(X_va).to(device))
    val_probs = torch.sigmoid(val_logits).cpu().numpy()

  roc_scores.append(roc_auc_score(y_va, val_probs))
  pr_scores.append(average_precision_score(y_va, val_probs))

print(
    f"{'PyTorch MLP':20s} | 5-fold ROC-AUC: {np.mean(roc_scores):.4f} +/-"
    f" {np.std(roc_scores):.4f} | PR-AUC: {np.mean(pr_scores):.4f} +/-"
    f" {np.std(pr_scores):.4f}"
)

Training device: cuda
PyTorch MLP          | 5-fold ROC-AUC: 0.7530 +/- 0.0380 | PR-AUC: 0.3070 +/- 0.0362


In [5]:
from collections import defaultdict
import numpy as np
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


def get_scaffold_smiles(mol):
  core = MurckoScaffold.GetScaffoldForMol(mol)
  return MurckoScaffold.MurckoScaffoldSmiles(mol=core, includeChirality=False)


def scaffold_train_test_split(mols, test_size=0.2):
  scaffold_to_indices = defaultdict(list)
  for idx, mol in enumerate(mols):
    scaf = get_scaffold_smiles(mol)
    scaffold_to_indices[scaf].append(idx)

  # Sort largest scaffold groups first so test set stays close to target size
  groups = sorted(
      scaffold_to_indices.values(), key=lambda g: len(g), reverse=True
  )

  n_total = len(mols)
  test_target = int(round(n_total * test_size))
  train_idx, test_idx = [], []

  for group in groups:
    if len(test_idx) + len(group) <= test_target:
      test_idx.extend(group)
    else:
      train_idx.extend(group)

  return np.array(train_idx, dtype=np.int64), np.array(
      test_idx, dtype=np.int64
  )


train_idx, test_idx = scaffold_train_test_split(p53_df["mol"], test_size=0.2)
print(
    f"Train size: {len(train_idx)} ({len(train_idx)/len(p53_df):.1%}) | Positives:"
    f" {y[train_idx].sum()} ({y[train_idx].mean():.2%})"
)
print(
    f"Test size:  {len(test_idx)} ({len(test_idx)/len(p53_df):.1%}) | Positives:"
    f" {y[test_idx].sum()} ({y[test_idx].mean():.2%})"
)

X_tr, y_tr = X[train_idx], y[train_idx]
X_te, y_te = X[test_idx], y[test_idx]

# 1. Logistic Regression & Random Forest on scaffold split
sk_models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42
    ),
}

print("\n--- Scaffold Split Results ---")
for name, clf in sk_models.items():
  clf.fit(X_tr, y_tr)
  probs = clf.predict_proba(X_te)[:, 1]
  print(
      f"{name:20s} | Test ROC-AUC: {roc_auc_score(y_te, probs):.4f} | Test"
      f" PR-AUC: {average_precision_score(y_te, probs):.4f}"
  )

# 2. PyTorch MLP on the same scaffold split
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)


class FingerprintMLP(nn.Module):

  def __init__(self, in_dim=2048):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(in_dim, 512),
        nn.BatchNorm1d(512),
        nn.SiLU(),
        nn.Dropout(0.3),
        nn.Linear(512, 128),
        nn.BatchNorm1d(128),
        nn.SiLU(),
        nn.Dropout(0.2),
        nn.Linear(128, 1),
    )

  def forward(self, x):
    return self.net(x).squeeze(-1)


n_pos = max(1.0, float(y_tr.sum()))
n_neg = max(1.0, float(len(y_tr) - y_tr.sum()))
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32, device=device)

train_ds = TensorDataset(
    torch.from_numpy(X_tr), torch.from_numpy(y_tr.astype(np.float32))
)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

mlp = FingerprintMLP(in_dim=X.shape[1]).to(device)
optimizer = torch.optim.AdamW(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

for epoch in range(25):
  mlp.train()
  for bx, by in train_loader:
    bx, by = bx.to(device), by.to(device)
    optimizer.zero_grad()
    loss = criterion(mlp(bx), by)
    loss.backward()
    optimizer.step()

mlp.eval()
with torch.no_grad():
  te_probs = (
      torch.sigmoid(mlp(torch.from_numpy(X_te).to(device))).cpu().numpy()
  )

print(
    f"{'PyTorch MLP':20s} | Test ROC-AUC: {roc_auc_score(y_te, te_probs):.4f} |"
    f" Test PR-AUC: {average_precision_score(y_te, te_probs):.4f}"
)

Train size: 5414 (80.0%) | Positives: 365 (6.74%)
Test size:  1353 (20.0%) | Positives: 58 (4.29%)

--- Scaffold Split Results ---
LogisticRegression   | Test ROC-AUC: 0.7561 | Test PR-AUC: 0.2245
RandomForest         | Test ROC-AUC: 0.7151 | Test PR-AUC: 0.1250
PyTorch MLP          | Test ROC-AUC: 0.7785 | Test PR-AUC: 0.2542


In [7]:
from rdkit import Chem
from rdkit.Chem import Descriptors

# Search by name in original df (if present) or match canonical SMILES for pentachlorophenol
pcp_smiles = "Oc1c(Cl)c(Cl)c(Cl)c(Cl)c1Cl"
pcp_mol = Chem.MolFromSmiles(pcp_smiles)
pcp_canon = Chem.MolToSmiles(pcp_mol)

# Check matches in cleaned p53_df and full df
p53_df["canon_smiles"] = p53_df["mol"].apply(
    lambda m: Chem.MolToSmiles(m, canonical=True)
)
matches_clean = p53_df[p53_df["canon_smiles"] == pcp_canon]

print(f"Canonical SMILES: {pcp_canon}")
print(f"Molecular weight: {Descriptors.MolWt(pcp_mol):.2f} g/mol")
print(f"Matches in clean SR-p53 dataset: {len(matches_clean)}")

if len(matches_clean) > 0:
  for idx, row in matches_clean.iterrows():
    x_vec = X[idx : idx + 1]
    lr_prob = sk_models["LogisticRegression"].predict_proba(x_vec)[0, 1]
    rf_prob = sk_models["RandomForest"].predict_proba(x_vec)[0, 1]

    mlp.eval()
    with torch.no_grad():
      mlp_prob = (
          torch.sigmoid(mlp(torch.from_numpy(x_vec).to(device))).cpu().item()
      )

    split_group = "TEST (unseen scaffold)" if idx in test_idx else "TRAIN"
    print(f"\nRow {idx} | mol_id: {row['mol_id']} | Split: {split_group}")
    print(f"  True SR-p53 label : {row['SR-p53']}")
    print(f"  LogReg pred prob  : {lr_prob:.4f}")
    print(f"  RandForest pred prob: {rf_prob:.4f}")
    print(f"  PyTorch MLP prob  : {mlp_prob:.4f}")
else:
  # Score pentachlorophenol directly even if absent from SR-p53
  x_pcp = mfp_gen.GetFingerprintAsNumPy(pcp_mol).astype(np.float32).reshape(1, -1)
  lr_prob = sk_models["LogisticRegression"].predict_proba(x_pcp)[0, 1]
  rf_prob = sk_models["RandomForest"].predict_proba(x_pcp)[0, 1]

  mlp.eval()
  with torch.no_grad():
    mlp_prob = (
        torch.sigmoid(mlp(torch.from_numpy(x_pcp).to(device))).cpu().item()
    )

  print("Predicted probabilities for pentachlorophenol:")
  print(f"  LogReg pred prob  : {lr_prob:.4f}")
  print(f"  RandForest pred prob: {rf_prob:.4f}")
  print(f"  PyTorch MLP prob  : {mlp_prob:.4f}")

Canonical SMILES: Oc1c(Cl)c(Cl)c(Cl)c(Cl)c1Cl
Molecular weight: 266.34 g/mol
Matches in clean SR-p53 dataset: 1

Row 27 | mol_id: TOX1106 | Split: TEST (unseen scaffold)
  True SR-p53 label : 1
  LogReg pred prob  : 0.4091
  RandForest pred prob: 0.2033
  PyTorch MLP prob  : 0.9756
